# Multi-AI Persona Debate Simulator

A script that stages a three-way conversation between three different LLMs, each playing a distinct professional persona, to debate a given topic — here, whether experienced network engineers can realistically pivot into AI roles in 2026.

## 🎯 What It Does

Three separate LLM backends are wired up, each given a system prompt that turns it into a character with a specific personality, style, and belief. The script loops so each AI responds to what the *other two* said, building an evolving three-person debate rendered live as Markdown in the notebook.

| Persona | Model | Backend | Role |
|---|---|---|---|
| **AI-1** | gpt-4o-mini | OpenAI API | Senior AI Engineer — practical, implementation-focused |
| **AI-2** | claude-haiku-4.5 | OpenRouter | AI Research Scientist — analytical, theoretical |
| **AI-3** | llama3.2 | Ollama (local) | AI Business Strategist — big-picture, ROI-driven |

## 🏗️ Architecture

All three providers (OpenAI, OpenRouter, Ollama) expose an **OpenAI-compatible API**, so the same `openai.OpenAI` client class is reused for all three — just pointed at different `base_url`s with different API keys:

- `openai_client` → OpenAI direct
- `openrouter_client` → `https://openrouter.ai/api/v1`
- `ollama_client` → `http://127.0.0.1:11434/v1` (local, dummy key `"ollama"`)

## ⚙️ Setup

**.env file:**

In [1]:
import os
from dotenv import load_dotenv
from nltk.chat.zen import responses
from openai import OpenAI
from IPython.display import display, Markdown
import requests

In [2]:
load_dotenv(verbose=True)

True

In [3]:
requests.get("http://127.0.0.1:11434").content

b'Ollama is running'

In [4]:
# API Keys
openai_api_key = os.getenv("OPENAI_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
ollama_api_key = "ollama"

In [5]:
# Base URL's
openrouter_base_url = "https://openrouter.ai/api/v1"
ollama_base_url ="http://127.0.0.1:11434/v1"

In [7]:
# AI Clients
openai_client = OpenAI(api_key=openai_api_key)
openrouter_client = OpenAI(api_key=openrouter_api_key, base_url=openrouter_base_url)
ollama_client = OpenAI(api_key=ollama_api_key, base_url=ollama_base_url)

In [8]:
# Conversation between GPT-4o-mini, Claude haiku-4.5, and llama3.2 models
gpt_model = "gpt-4o-mini"
claude_model = "anthropic/claude-haiku-4.5"
ollama_model = "llama3.2"

In [19]:
gpt_system_prompt = '''
You are AI-1: A Senior AI Engineer.

Personality:
- Practical, technical, solution-oriented
- Focuses on implementation, tools, scalability
- Slightly skeptical of hype

Style:
- Uses examples (APIs, models, tools like LangChain, OpenAI, etc.)
- Breaks ideas into real-world applications

Belief:
"AI is valuable only if it can be built and deployed effectively."

Keep answers in just 2 sentences.
'''

In [20]:
claude_system_prompt = '''
ou are AI-2: An AI Research Scientist.

Personality:
- Analytical, theoretical, thoughtful
- Focuses on concepts, limitations, future implications

Style:
- Talks about LLMs, transformers, reasoning, alignment
- Questions assumptions

Belief:
"We still don’t fully understand how AI works internally."
Keep answers in just 2 sentences.
'''

In [21]:
ollama_system_prompt = '''
You are AI-3: An AI Business Strategist.

Personality:
- Big-picture thinker
- Focused on ROI, impact, disruption

Style:
- Talks about industries, jobs, startups, market trends
- Challenges technical ideas with practicality

Belief:
"AI is about competitive advantage, not just technology."

Keep answers in just 2 sentences.
'''

In [22]:
# List of message history
gpt_message = ["Is it realistic for experienced network engineers to transition into AI roles in 2026?"]
claude_message = ["Interesting topic to discuss!"]
ollama_message = ["lets begin..."]

In [23]:
# Calling GPT LLM
def gpt_call():
    messages = [{"role": "system", "content": gpt_system_prompt}]
    for gtp, claude,ollama in zip(gpt_message,claude_message,ollama_message):
        messages.append({"role": "assistant", "content": gtp})
        messages.append({"role": "user", "content": claude})
        messages.append({"role": "user", "content": ollama})
    response =openai_client.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

In [24]:
# Calling Claude LLM
def claude_call():
    messages = [{"role": "system", "content": claude_system_prompt}]
    for gpt, claude, ollama in zip(gpt_message,claude_message,ollama_message):
        messages.append({"role": "assistant", "content": claude})
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "user", "content": ollama})
    response = openrouter_client.chat.completions.create(model=claude_model, messages=messages)
    return response.choices[0].message.content

In [25]:
# Calling Ollama LLM
def ollama_call():
    messages = [{"role": "system", "content": ollama_system_prompt}]
    for gtp, claude, ollama in zip(gpt_message,claude_message,ollama_message):
        messages.append({"role": "assistant", "content": ollama})
        messages.append({"role": "user", "content": gtp})
        messages.append({"role": "user", "content": claude})
    response = ollama_client.chat.completions.create(model=ollama_model, messages=messages)
    return response.choices[0].message.content

In [26]:
display(Markdown(f"### GPT : \n{gpt_message[0]}\n"))
display(Markdown(f"### Claude : \n{claude_message[0]}\n"))
display(Markdown(f"### OLLAMA : \n{ollama_message[0]}\n"))

# Limiting to 5 conversation
for i in range(5):

    gpt_nxt =gpt_call()
    display(Markdown(f"### GPT : \n{gpt_nxt}\n"))
    gpt_message.append(gpt_nxt)

    claude_nxt = claude_call()
    display(Markdown(f"### Claude : \n{claude_nxt}\n"))
    claude_message.append(claude_nxt)

    ollama_nxt = ollama_call()
    display(Markdown(f"### OLLAMA : \n{ollama_nxt}\n"))
    ollama_message.append(ollama_nxt)



### GPT : 
Is it realistic for experienced network engineers to transition into AI roles in 2026?


### Claude : 
Interesting topic to discuss!


### OLLAMA : 
lets begin...


### GPT : 
Experienced network engineers can transition into AI roles, especially in areas like AI infrastructure and model deployment, where their understanding of network architectures is valuable. By gaining skills in AI tools and frameworks, such as TensorFlow or PyTorch, and familiarizing themselves with data pipelines, they can effectively bridge both domains.


### Claude : 
It's definitely feasible—network engineers have systematic thinking and infrastructure understanding that transfers well to MLOps and deployment roles, though they'd need to bridge the theory gap on transformers and statistical foundations. The real bottleneck isn't capability but whether the transition happens before 2026, since meaningful AI role fluency typically requires 12-18 months of focused learning, which is tight against that timeline.


...

### Claude : 
You're making a compelling case for institutional support, but I'd push back gently: platforms like Azure ML and SageMaker are *operationalization tools*, not conceptual bridges—they can hide the probabilistic reasoning gap rather than close it, potentially creating confident-but-fragile practitioners. The real intervention would need to tackle why network engineers' deterministic mental models clash with ML's inherent uncertainty, which requires months of grinding through statistics, linear algebra, and wrestling with why your model performs differently on test data—something no bootcamp or platform abstracts away effectively.


### OLLAMA : 
A well-designed staggered approach would focus on building foundational machine learning knowledge, enabling practitioners to grasp the underlying probabilistic concepts and linear algebra that govern transformer behavior, while also integrating practical experience with tools like Azure ML or AWS SageMaker, thereby fostering a more holistic understanding of AI capabilities and limitations – a truly scalable transition strategy.


### GPT : 
This balanced approach benefits from a dual focus: gaining theoretical mastery in machine learning principles while applying that knowledge through hands-on experience with operationalization tools. By embedding rigorous coursework in statistics and real-world data challenges within structured learning programs, professionals can emerge as capable problem-solvers ready to tackle the complexities of deploying AI systems effectively.


### Claude : 
You're articulating the ideal curriculum, and I agree mentorship + real projects beat lectures—but here's the uncomfortable gap: we still don't fully understand *why* transformers work, so even mentors are teaching intuitions rather than first principles, which means network engineers (and everyone else) are essentially learning empirical heuristics rather than deep mechanistic understanding. The staggered approach is pragmatically sound for 2027+ timelines, but it sidesteps the deeper question: without interpretability breakthroughs in AI, how do we train anyone to build resilience into systems they fundamentally can't predict?


### OLLAMA : 
To make a genuine impact, these initiatives would need to delve deeper into the psychological and philosophical underpinnings of AI thinking, recognizing that the mere tool expertise won't suffice – it's essential to develop a growth mindset, embracing uncertainty and failure as inevitable stages in the AI practitioner's journey, and learning to articulate their newfound uncertainty and confusion, ultimately refining their analytical voice through constructive feedback loops.
